In [48]:
import tensorflow as tf
print(tf.__version__)


2.14.0


In [49]:
import os
import VGG_help1
from VGG_help1 import prepare_dataset, test_on_data, plot_train_history, plot_confusion_matrix, analyze_performance, train_and_evaluate_from_arrays
from VGG_help1 import vgg_model, resnet_model, cv_train_and_evaluate_model, train_and_evaluate_model, cv_train_model, cv_train_vgg_model, imbalanced_cv_train_and_evaluate_model, custom_cnn

In [50]:
import importlib
import VGG_help1
importlib.reload(VGG_help1)


<module 'VGG_help1' from 'c:\\Users\\ZINGA\\Documents\\DDPM_X-Ray-main.21.02.2025\\Codes\\Classification_Models.1.0\\VGG_help1.py'>

#### Usando somente os dados sinteticos

In [ ]:

project_root = r'C:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\project_root'

dataset_dir_generated = os.path.join(project_root, 'generated', 'train')


In [52]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import math
import os

# ============================
# PARÂMETROS (garantir escopo)
# ============================
batch_size = 32
img_size = (128, 128)
seed = 42

# Path dos dados sintéticos
project_root = r'C:\Users\ZINGA\Documents\DDPM_X-Ray-main.21.02.2025\project_root'
dataset_dir_generated = os.path.join(project_root, 'generated', 'train')

# ============================
# SPLIT 70 / 15 / 15
# ============================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.30
)

# -------- TRAIN (70%) --------
train_gen = datagen.flow_from_directory(
    dataset_dir_generated,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=seed
)

# -------- TEMP (30%) --------
temp_gen = datagen.flow_from_directory(
    dataset_dir_generated,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=seed
)

# ============================
# TEMP → VAL + TEST
# ============================
X_temp, y_temp = [], []
steps = math.ceil(temp_gen.samples / batch_size)

for i in range(steps):
    x, y = temp_gen[i]
    X_temp.append(x)
    y_temp.append(y)

X_temp = np.concatenate(X_temp)
y_temp = np.concatenate(y_temp)

split_idx = len(X_temp) // 2

X_val, X_test = X_temp[:split_idx], X_temp[split_idx:]
y_val, y_test = y_temp[:split_idx], y_temp[split_idx:]


Found 560 images belonging to 2 classes.
Found 240 images belonging to 2 classes.


In [53]:
print("Train samples:", train_gen.samples)
print("Val samples:", len(X_val))
print("Test samples:", len(X_test))
print("Classes:", train_gen.class_indices)


Train samples: 560
Val samples: 120
Test samples: 120
Classes: {'NORMAL': 0, 'PNEUMONIA': 1}


In [54]:
# ============================
# Extrair X_train e y_train do generator
# ============================
X_train, y_train = [], []

steps_train = math.ceil(train_gen.samples / batch_size)
for i in range(steps_train):
    x, y = train_gen[i]
    X_train.append(x)
    y_train.append(y)

X_train = np.concatenate(X_train)
y_train = np.concatenate(y_train)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (560, 128, 128, 3)
y_train shape: (560, 2)


#### Desenet212

In [59]:
from tensorflow.keras.utils import to_categorical

# Garantir one-hot (se necessário)
if y_train.ndim == 1:
    y_train = to_categorical(y_train, num_classes)
if y_val.ndim == 1:
    y_val = to_categorical(y_val, num_classes)
if y_test.ndim == 1:
    y_test = to_categorical(y_test, num_classes)

metrics_runs_densenet = []

for i in range(n):
    print(f"DenseNet Run {i+1}/{n}")


    trained_model_densenet, test_metrics_densenet, confusion_matrix_densenet = (
    VGG_help1.train_and_evaluate_from_arrays(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        model_fn=VGG_help1.build_densenet_model,
        input_shape=input_shape,
        num_classes=num_classes,
        epochs=epochs,
        batch_size=batch_size,
        title=f"{title_densenet}_Run{i+1}"
    )
)


    trained_model_densenet.save(
        f"Trained_Models/{title_densenet}_run{i+1}.h5"
    )

    test_metrics_densenet["Config"] = "SyntheticOnly"
    test_metrics_densenet["Run"] = i + 1
    metrics_runs_densenet.append(test_metrics_densenet)

    print(f"Run {i+1} Metrics:", test_metrics_densenet)



DenseNet Run 1/5
Epoch 1/5
 4/18 [=====>........................] - ETA: 58s - loss: 1.0245 - accuracy: 0.5469 

KeyboardInterrupt: 

In [38]:
print(hasattr(VGG_help1, "train_and_evaluate_from_arrays"))


True


In [57]:
import numpy as np

# Extrair apenas as acurácias dos testes
test_accuracies = [m["Test Accuracy"] for m in metrics_runs_densenet]

# Estatísticas gerais
mean_acc = np.mean(test_accuracies)
std_acc = np.std(test_accuracies)

print("=== Estatísticas Gerais de Test Accuracy ===")
print(f"Acurácia média: {mean_acc:.4f}")
print(f"Desvio padrão: {std_acc:.4f}")
print(f"Melhor acurácia: {np.max(test_accuracies):.4f}")
print(f"Pior acurácia: {np.min(test_accuracies):.4f}")


=== Estatísticas Gerais de Test Accuracy ===
Acurácia média: 0.4333
Desvio padrão: 0.3212
Melhor acurácia: 1.0000
Pior acurácia: 0.0917


In [58]:
test_losses = [m["Test Loss"] for m in metrics_runs_densenet]
print("=== Estatísticas Gerais de Test Loss ===")
print(f"Loss médio: {np.mean(test_losses):.4f}")
print(f"Desvio padrão: {np.std(test_losses):.4f}")
print(f"Menor loss: {np.min(test_losses):.4f}")
print(f"Maior loss: {np.max(test_losses):.4f}")



=== Estatísticas Gerais de Test Loss ===
Loss médio: 1.3161
Desvio padrão: 1.0889
Menor loss: 0.0728
Maior loss: 3.2896


# Train on Selected Data

### ResNet50 pre-treined

### ResNet50 pre-treined

### VGG16 pre-treined